# Layer 6 — Sanity Check

Read-only: imports `src.*` and reads Layer 1-5 CSV output. No computation happens here -- if a number looks wrong, the bug is upstream in `scripts/`, not in this notebook.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from src.config import PROCESSED_DIR, RESULTS_DIR, N_SPLITS

## Layer 1 — preprocessing

In [2]:
manifest = json.load(open(PROCESSED_DIR / 'manifest.json'))
print(manifest)
fold_summary = pd.read_csv(PROCESSED_DIR / 'fold_summary.csv')
fold_summary

{'min_user_ratings': 5, 'min_movie_ratings': 10, 'random_state': 42, 'n_splits': 10, 'outlier_threshold': 2.0, 'total_rows_after_filter': 97953, 'n_users': 943, 'n_movies': 1152, 'max_user_id': 943, 'max_movie_id': 1615}


,fold,train_raw,train_clean,outliers_removed,train_inner,val_inner,test_raw,test_norm
0,0,87724,84343,3381,75502,8841,10229,10229
1,1,87820,84502,3318,75650,8852,10133,10133
2,2,87934,84570,3364,75713,8857,10019,10019
3,3,88036,84686,3350,75815,8871,9917,9917
4,4,88122,84927,3195,76030,8897,9831,9831
5,5,88209,84855,3354,75966,8889,9744,9744
6,6,88302,84969,3333,76070,8899,9651,9651
7,7,88397,84955,3442,76049,8906,9556,9556
8,8,88474,85199,3275,76274,8925,9479,9479
9,9,88559,85154,3405,76233,8921,9394,9394


## Layer 2 — SVD++ / DecayPop

In [3]:
rmse_summary = pd.read_csv(RESULTS_DIR / 'svdpp' / 'rmse_summary.csv')
rmse_summary

,fold,best_val_rmse,final_test_rmse,epochs_run
0,0,0.839638,0.911876,14
1,1,0.828029,0.915952,14
2,2,0.840732,0.923452,14
3,3,0.837617,0.930525,13
4,4,0.831696,0.919419,13
5,5,0.838805,0.922766,15
6,6,0.840075,0.927885,13
7,7,0.840015,0.923359,17
8,8,0.843109,0.929105,13
9,9,0.840490,0.943086,14


In [4]:
decaypop_normalized = pd.read_csv(RESULTS_DIR / 'decaypop' / 'decaypop_normalized.csv')
decaypop_normalized.describe()

,movie_id,decaypop_raw,decaypop_normalized
count,1152.000000,1152.000000,1.152000e+03
mean,619.582465,0.015033,-3.700743e-17
std,377.604623,2.464552,1.000434e+00
min,1.000000,-7.534351,-3.064518e+00
25%,298.750000,-1.114079,-4.583400e-01
50%,594.000000,-0.327030,-1.388536e-01
75%,932.250000,0.543535,2.145344e-01
max,1615.000000,21.334541,8.654216e+00


## Layer 4 — metrics: rasio terbaik per protokol

In [5]:
metrics_fc = pd.read_csv(RESULTS_DIR / 'metrics' / 'metrics_fullcatalog.csv')
metrics_fc.sort_values('ndcg_mean', ascending=False)[['ratio', 'precision_mean', 'ndcg_mean']]

,ratio,precision_mean,ndcg_mean
0,10_90,0.058258,0.086577
1,20_80,0.056889,0.083203
2,30_70,0.055394,0.080636
3,40_60,0.054157,0.078594
4,50_50,0.052767,0.076335
5,60_40,0.050852,0.073760
6,70_30,0.049859,0.072020
7,80_20,0.048711,0.070289
8,90_10,0.047015,0.067584
9,SVD_only,0.043671,0.063701


In [6]:
metrics_to = pd.read_csv(RESULTS_DIR / 'metrics' / 'metrics_testonly.csv')
metrics_to[metrics_to['split'] == 'test_final'].sort_values('ndcg_mean', ascending=False)[['ratio', 'precision_mean', 'ndcg_mean']]

,ratio,precision_mean,ndcg_mean
16,90_10,1.482435,0.874213
14,80_20,1.625862,0.873886
18,SVD_only,1.284287,0.873828
12,70_30,1.668469,0.872442
10,60_40,1.526456,0.870970
8,50_50,1.335489,0.868222
6,40_60,1.165389,0.865566
4,30_70,0.985890,0.862047
2,20_80,0.873414,0.857711
0,10_90,0.789574,0.852606


## Layer 5 — segmentasi

In [7]:
segments = pd.read_csv(RESULTS_DIR / 'segmentation' / 'user_segments.csv')
segments['segment'].value_counts(normalize=True)

segment
regular    0.694486
new        0.160870
trend      0.144645
Name: proportion, dtype: float64